# 01 Single Logic Run

Run one symbol/timeframe with one fixed execution config. Use this notebook to inspect signal loading, next-open entry, SL/TP levels, cluster behavior, skip reasons, ambiguity, and cluster-level R metrics.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
project_root = cwd
while project_root.name != "SEN05" and project_root.parent != project_root:
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

BACKTEST_ROOT = project_root / "backtest_optimize"
RAW_SIGNALS = project_root / "raw_signals"
OUTPUT_DIR = BACKTEST_ROOT / "outputs" / "single_runs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

project_root

In [ ]:
import pandas as pd

from backtest_optimize.contracts import AmbiguityPolicy, MarketSpec
from backtest_optimize.io.signal_loader import load_signal_csv
from backtest_optimize.io.market_data import load_ohlcv_from_core
from backtest_optimize.execution.engine import run_single
from backtest_optimize.analysis.metrics import clusters_to_frame, summarize
from backtest_optimize.analysis.versioning import save_snapshot

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

In [ ]:
# Required user inputs. Do not infer symbol/timeframe from filename.
SYMBOL = "US30"
TIMEFRAME = "H4"
SIGNAL_FILE = RAW_SIGNALS / "combo" / "combo_US30_H4_20230102_20260511_signals.csv"
WARMUP_BARS = 0  # Increase when using swing_extreme SL or other lookback-based methods.

# Adjust market spec per symbol before trusting R/cost output.
MARKET_SPEC = MarketSpec(
    symbol=SYMBOL,
    pip_size=1.0,
    pip_value_per_lot=1.0,
    min_lot=0.01,
    lot_step=0.01,
    commission_per_lot_per_side=0.0,
    spread_buffer_pips=0.0,
    slippage_buffer_pips=0.0,
)

RUN_CONFIG = {
    "account_size": 10_000.0,
    "risk_per_cluster": 0.01,
    "sl_method": "atr_multiple",
    "sl_params": {"atr_mult": 1.5},
    "tp_method": "risk_multiple",
    "tp_params": {"r_multiples": [1.0, 2.0, 3.0]},
    "ambiguity_policy": AmbiguityPolicy.CONSERVATIVE,
    "management": {"sl_move_rule": "breakeven_after_tp1"},
}

SIGNAL_FILE

In [ ]:
signals = load_signal_csv(SIGNAL_FILE, symbol=SYMBOL, timeframe=TIMEFRAME)

start = signals["bartime"].min()
end = signals["bartime"].max() + pd.Timedelta(days=10)
bars = load_ohlcv_from_core(SYMBOL, TIMEFRAME, start=start, end=end, warmup_bars=WARMUP_BARS, tail_bars=5)

print(f"signals: {len(signals):,}")
print(f"bars:    {len(bars):,}")
display(signals.head())
display(bars.head())

In [ ]:
result = run_single(
    signals=signals,
    bars=bars,
    symbol=SYMBOL,
    timeframe=TIMEFRAME,
    market_spec=MARKET_SPEC,
    **RUN_CONFIG,
)

summary = summarize(result)
clusters = clusters_to_frame(result)

display(pd.DataFrame([summary]))
display(clusters.head(20))

In [ ]:
run_name = f"single_{SYMBOL}_{TIMEFRAME}_{pd.Timestamp.now('UTC').strftime('%Y%m%d_%H%M%S')}"
clusters_path = OUTPUT_DIR / f"{run_name}_clusters.csv"
summary_path = OUTPUT_DIR / f"{run_name}_summary.csv"

clusters.to_csv(clusters_path, index=False)
pd.DataFrame([summary]).to_csv(summary_path, index=False)

snapshot_path = save_snapshot(
    name=run_name,
    config={**RUN_CONFIG, "market_spec": MARKET_SPEC, "symbol": SYMBOL, "timeframe": TIMEFRAME},
    result_summary=summary,
    signal_file=SIGNAL_FILE,
    market_data_source_id="core_python.data.loader",
    assumptions=result.assumptions,
    repo_root=project_root,
)

print(clusters_path)
print(summary_path)
print(snapshot_path)